# Audio Analysis Notes: Music Similarity, Speech, and Sound

Current focus: compare music and accompaniment across videos using timestamped audio windows and separate representations. This notebook documents options and evidence limits; the master workflow and completed run records are linked from the project README. Existing trial results are exploratory, not content or intent labels.

For now, prioritize music/audio vectors and optional speech-emotion representations. Defer VAD + ASR transcript semantics and general sound-event classification until a research question specifically needs spoken content or event labels.

## How should we define a piece or phase?

There are several different meanings of “segment,” and they should not be conflated:

- **Analysis frame/window:** a small, often overlapping interval (for example, 2 seconds with a 1-second hop) used to calculate features. This gives consistent input units for vectorizing and clustering; it does not claim a musical boundary.
- **Acoustic event/scene boundary:** a point where the sound changes, such as speech starting, music changing, a cut, or a loud effect. Voice activity detection (VAD), change-point detection, or source/event classifiers can help locate these.
- **Musical section/phase:** a verse, chorus, intro, or outro. Repetition and musical structure models can suggest these, but chroma alone cannot reliably name them. Human review or a trained section-labeling model is needed for confident labels.

For an initial Douyin workflow, do not require verse/chorus labels before analysis. First identify speech/music/silence and major acoustic changes; then create fixed or speech-aligned windows for comparable feature vectors. Keep timestamps so any result can be traced back to the original audio. If musical phases matter for a later question, run a separate music-structure step on music-dominant spans.

A practical starting point is 2–5 second windows with overlap, adjusted to the task. Short windows localize changes, while overlap reduces the chance that a key phrase or transition is split exactly at a boundary. For ASR and rhetoric analysis, sentence/utterance spans are often better than arbitrary fixed windows. There is no universally correct duration; check several examples and tune it to the content and model.

## Do the existing segment boundaries work?

The current [segment_boundaries.csv](../results/librosa/segment/segment_boundaries.csv) was produced in the clean-analysis notebook from **chroma self-similarity** using `librosa.segment.agglomerative(chroma, k=8)`. In other words, it proposes coarse divisions based on changes in pitch-class patterns, with the number of segments fixed to eight.

The saved boundary times are approximately 0.00, 3.70, 81.28, 102.82, 141.76, 163.87, 184.80, and 189.40 seconds. They are useful as an exploratory view of musical structure, but they are not yet validated phase labels. The widely varying intervals (including a roughly 77-second span) show that these are not suitable as short, comparable clustering windows. The last recorded point is not the audio end: the track summary reports about 205.93 seconds, so a downstream consumer must explicitly include the final tail through the audio duration.

The method does not detect speech, determine foreground/background, or know whether a boundary is a verse/chorus transition. Its fixed `k=8` is a modeling choice, not a discovered fact. Treat the output as candidate structural boundaries, inspect the recurrence plot and listen around candidate points, and compare against other tracks before using it. For Douyin, chroma can be weak or misleading when clips are speech-led, noisy, heavily edited, or contain little tonal music.

So: **yes, it works as a first coarse music-structure experiment; no, it is not enough as the segmentation stage for propaganda/audio-content clustering.** Build a separate timeline for speech activity and acoustic events, then choose windows suited to the speech or other target content.

## How can a machine tell foreground/main content from background or messy audio?

A machine does not inherently know what is important. “Main content” is a task definition, and the system estimates it from measurable cues and models. Also distinguish **foreground** (prominent or close in the mix) from **important content** (relevant to the research question): quiet speech can be important, while loud music may be background.

Useful signals and tools include:

1. **VAD (voice activity detection):** estimates which time intervals contain speech. It helps find speech candidates, but may miss singing, distant/overlapped speech, or distorted speech, and may mark speech-like sounds as speech.
2. **Audio event/source classification and separation:** estimates speech, music, noise, or sources such as vocals and accompaniment. Separation can make a source easier to analyze, but artifacts are possible; separated stems are estimates, not ground truth. A vocals stem can include singing and can leak music/noise.
3. **Acoustic quality measurements:** clipping, silence, signal-to-noise ratio (SNR), loudness, spectral shape, and abrupt changes can flag technical problems or unusual spans. These are quality cues, not reliable judgments of importance. Loudness alone cannot distinguish speech from loud background music.
4. **ASR and transcript checks:** speech recognition can reveal whether a speech candidate contains words and help align utterances. ASR can hallucinate or fail on music, noise, accents, and overlapping speakers, so retain confidence/timestamps and review uncertain samples.
5. **Human review and task labels:** sample clips should be listened to and labeled with a small, explicit scheme (for example speech, music, mixed, silence/noise, uncertain; plus whether speech is relevant). This is how we measure whether automatic “main content” decisions match the research goal.

A robust first pass is to keep the original audio and create timestamped candidate intervals; add VAD and speech/music estimates; optionally separate sources and run ASR; then assign confidence and review uncertain or high-impact spans. Do not automatically discard a span only because it is quiet, noisy, or classified as background: it may contain relevant speech or a meaningful sound cue. Preserve both the original and processed versions, and log processing choices.

## How to clean up messy audio

First decide what “messy” means for the intended analysis. Technical defects (clipping, hum, wind, low volume) differ from a meaningful mixture (speech over music, several speakers, or sound effects). Cleaning should make the target easier to analyze without erasing evidence.

Suggested order:

1. **Inspect and preserve:** keep an unchanged source copy; record duration, sample rate, channels, and provenance. Listen to representative clips and inspect waveform/spectrogram. Do not overwrite the source.
2. **Standardize safely:** convert to the expected sample rate/channel format for each model, check for empty/corrupt files, and avoid clipping during resampling or level adjustment.
3. **Measure and flag:** detect clipped samples, long silence, very low level, abrupt cuts, and likely low-SNR spans. Use flags and confidence rather than silently deleting them.
4. **Separate the task-relevant source:** use VAD for speech timing, speech enhancement for noisy speech, or source separation when speech and music overlap. Compare enhanced/separated results with the original; processing can distort words and transient sounds.
5. **Transcribe and review:** run ASR on speech candidates, align words/utterances to timestamps, and manually inspect uncertain or consequential spans. Keep “unintelligible/uncertain” as a valid result rather than forcing a guess.
6. **Create analysis windows and vectors:** extract features for each timestamped interval from the original and, where useful, from separated stems. Store provenance and quality indicators alongside every vector.

Avoid treating denoising as neutral: a model may suppress quiet voices, alarms, music, or other meaningful signals. For propaganda research, the content and delivery may both matter, so keep separate representations for speech/text, voice/prosody, music, and overall mix rather than cleaning everything into one signal. Compare downstream results with and without enhancement on a reviewed sample.

## Current workflow: implemented trial pipeline and planned next steps

The diagram separates work that already runs in the Billie Eilish trial from work that is still planned. The existing pipeline analyzes music/audio features; it does **not** yet detect propaganda, classify foreground/background/noise, create a unified segment-level vector, or cluster Douyin clips.

```text
IMPLEMENTED TRIAL (Billie Eilish audio)
YouTube/local audio
  -> use locally supplied video/audio; extract audio with FFmpeg
  -> Demucs source separation (vocals / no_vocals for bad guy trial)
       -> Basic Pitch on vocals (note events and MIDI)
  -> Librosa analysis of original and separated stems
       -> summary features: RMS, MFCC means, spectral contrast, tempo estimate
       -> chroma profile and key candidates
       -> coarse chroma self-similarity segmentation (fixed k=8)
       -> beat tracking
       -> DTW chroma comparison of original and no_vocals
  -> CSV, plots, stems, and MIDI in results/

PLANNED FOR THE DOUYIN RESEARCH PIPELINE (not implemented yet)
Douyin clips
  -> preserve original + inspect/measure quality
  -> estimate speech/music/events; review uncertain spans
  -> timestamped short windows or speech-aligned utterances
  -> feature/embedding vectors per window (+ text vectors if useful)
  -> scale/normalize and cluster; inspect examples and evaluate
```

The coarse chroma boundaries are candidate musical-structure boundaries only. They are not a substitute for speech/event detection or fixed, comparable analysis windows. The existing DTW output is an alignment/comparison result, not a clustering result.

## Common tools for vectorizing audio (and what each vector represents)

Vectorization means mapping a whole clip or a time window to a fixed-size list of numbers. There is no single universally correct audio vector: the representation determines what “similar” means. For this project, keep timestamps, clip IDs, quality flags, and the source of each vector next to it.

| Approach / common tools | What it produces | Useful for | Important limitation |
|---|---|---|---|
| **Handcrafted acoustic features** — `librosa`, `torchaudio`, Essentia | MFCC, RMS/loudness, spectral centroid/contrast, chroma, pitch, onset/rhythm; summarize each window with mean/std/quantiles | Interpretable acoustic comparisons; a practical baseline and quality checks | Requires feature and window choices; music-oriented features may not represent speech meaning or propaganda rhetoric. MFCC/chroma summaries already exist in the trial, but not yet as a row-per-window clustering matrix. |
| **General audio embeddings** — YAMNet, PANNs, VGGish, OpenL3 | Pretrained neural vectors, often one vector per short frame/window; aggregate frame vectors if a clip-level vector is needed | Broad sound/event or timbral similarity; useful when manual features are insufficient | Models differ in training data, labels, input duration, and intended use. YAMNet is also commonly used for event scores; score outputs are not the same thing as embeddings. Check licensing, model availability, and performance on target clips. |
| **Audio-text embeddings** — CLAP (Contrastive Language-Audio Pretraining) | Audio and text mapped into a shared embedding space | Comparing audio to text descriptions, or exploring broad semantic sound similarity | Similarity to prompts is not a verified content label, and not a reliable standalone propaganda detector. |
| **Speech representations** — wav2vec 2.0, HuBERT, WavLM via Hugging Face/`torchaudio` | Frame-level learned speech representations; pool over an utterance/window for fixed-size vectors | Speech sound/phonetic and voice-delivery patterns | Often requires speech-focused preprocessing and pooling; vectors are not transcript meaning and can encode speaker/channel effects. |
| **Speaker embeddings** — SpeechBrain ECAPA-TDNN, pyannote.audio | A vector intended to represent speaker identity/voice characteristics | Diarization or comparing voice identity across utterances, when appropriate | This is not a topic/content vector; handle identity-related data carefully, and do not infer identity from a cluster without validation. |
| **Speech-to-text then text embeddings** — Whisper/faster-whisper for ASR; Sentence-Transformers for sentence vectors | ASR text with timestamps, then a text vector per utterance or sentence | Topic, claims, repeated slogans, and rhetorical/semantic similarity | ASR errors propagate; text vectors omit vocal delivery, music, and acoustic context. Keep ASR confidence and review uncertain transcripts. |

`librosa` and `torchaudio` are feature/audio-processing libraries, not one-click semantic vectorizers. In the current notebook, `librosa` is already used for handcrafted features; the other model families above are options to evaluate, not tools currently wired into this project.

A sensible baseline is to begin with per-window handcrafted features and one pretrained audio embedding model, then compare results on a small manually reviewed sample. Add ASR plus text embeddings if the research question concerns spoken claims or rhetoric. Keep audio, speech-delivery, speaker, and text representations as separate feature groups initially; compare them individually before fusing, because concatenating unscaled vectors can let one group dominate. Scale numeric features, handle missing values, reduce dimensions only when useful, and use the same preprocessing consistently for every clip. Evaluate clusters by listening to representative and boundary cases, not only by a numeric clustering score.

## Current analysis plan: audio vectors and cross-video music similarity

A vector records a model's or feature extractor's representation of an audio window. Similarity is always relative to that representation: two windows can be acoustically similar without sharing meaning, source, or intent. Keep each vector linked to its video ID, track/stem, start/end time, title/source metadata, and processing settings.

| Analysis question | Method | Output and interpretation |
|---|---|---|
| How does the voice sound emotionally? | **emotion2vec** on speech-bearing crops from `vocals` or the original mix | Speech-emotion representation vectors; use for comparing delivery patterns. A base representation model does not necessarily provide calibrated emotion labels. Any anger/sadness/etc. classifier needs language/domain evaluation and human review; this is not access to a speaker's true internal state. |
| How does the sound change acoustically? | **Librosa** on consistent windows and stems | Interpretable MFCC, RMS/energy, pitch, zero-crossing rate, spectral centroid/bandwidth/contrast, chroma, and rhythm features. These describe measurable sound and can support comparisons; they do not directly classify feelings or semantics. |
| Which music passages sound similar across videos? | **MERT embeddings**, preferably on music-dominant / `no_vocals` windows; Librosa rhythm/chroma/timbre features as interpretable companion views | One vector per timestamped music window, enabling cosine-similarity nearest neighbors and clustering. Inspect retrieved pairs by listening. MERT is music-oriented, but it may still encode recording, production, loudness, and source-video differences. |
| Does audio resemble a textual sound description? | **CLAP** audio-text similarity | Scores/rankings against carefully balanced text prompts (for example, “solemn orchestral music” and comparison descriptions). Treat scores as exploratory retrieval cues, not verified tags; check Chinese-language support and wording. |
| What is being said, and what topic does it concern? | **VAD + ASR + text embeddings** | Speech regions, timestamped transcript, and semantic text vectors. **Deferred for now**; bring this in when spoken claims/rhetoric become the target question. |
| Which non-speech events occur? | **Audio event recognition** | Candidate event labels such as applause, crowd cheer, or alarm. **Deferred for now**; add only if event detection becomes a specific research objective. |

### A practical design for finding music similarities across videos

1. **Define the comparison unit.** Compare music passages to music passages, not arbitrary full videos. Use a consistent window/model setting (the current baseline is 5 s windows); exclude incomplete padded tail windows from clustering. Overlapping windows are useful for locating changes, but they are correlated observations, not independent examples.
2. **Separate or select the music-bearing audio carefully.** Demucs `no_vocals` means the model-estimated accompaniment/non-vocal stem, not guaranteed pure music. It can retain speech/crowds or contain separation artifacts. Retain the original and inspect representative crops; optionally use human listening to mark music-dominant spans. VAD+ASR is not required if the immediate question is music similarity.
3. **Start with MERT for learned music similarity.** L2-normalize vectors and use cosine similarity to retrieve nearest windows across *different* videos. Keep same-video neighbors separately as a within-track baseline. Examine matches at several thresholds/ranks rather than treating a single score as a probability.
4. **Use Librosa as a complementary explanation, not a replacement label.** Compare tempo/onset patterns, chroma (pitch-class/harmonic patterns), MFCC/spectral timbre, and energy. If MERT finds a cross-video pair, these features can help describe whether it shares rhythm, harmony, or timbre. Tempo/key/chroma estimates can be unreliable on short, noisy, speech-heavy, or non-tonal clips.
5. **Control source and sampling effects.** Give every video equal weight in corpus comparisons; avoid letting long videos dominate. Check whether clusters correspond to video/source, encoding, loudness, or the Demucs stem rather than music. The current three-video balanced k=3 experiment separated by source video, so it did **not** establish cross-video shared music groups.
6. **Validate with listening and metadata.** Review top nearest-neighbor pairs, distant/borderline pairs, and examples from each cluster. Record whether the match sounds like the same recording/sample, a cover/remix, or only a broadly similar style. Titles can help organize/review candidates, but should not be treated as ground-truth music labels.

For robust discovery, a useful corpus should contain many independent videos, multiple examples of recurring music, and enough metadata to identify duplicates/reposts. Split evaluation by video/source, not by overlapping windows. A cluster or high cosine score only says “close under this representation”; it does not prove identical music, a common topic, propaganda, or intent.

**Scope now:** prioritize MERT + Librosa for music-window similarity, and optionally emotion2vec for speech delivery after speech crops are available. **Not currently needed:** VAD/ASR text semantics and general sound-event labels. Add those only when the research question requires what was spoken or which events occurred.

## 音乐情绪编码：框架、数据与 MERT/Librosa 的使用方法

本节是后续研究方案，不表示这些情绪模型或预测器已经在项目音频上运行。现有 MERT/Librosa 分析可以作为候选输入；情绪预测必须先有合适的人工评分或标签，并在独立歌曲/视频上验证。

### 1. 先定义要测量的“情绪”

- **音乐表达的情绪**：音乐听起来像在表达什么，例如悲伤、庄严或欢快。
- **听众实际感受到的情绪**：听者听完后报告自己感到什么。它会受个人、文化、情境和听音方式影响，不一定与音乐表达一致。

开始前选定其中一个目标，并把问题、评分说明和标注对象写进标注指南。模型只能预测与训练标签相近的评分，不能从录音证实听众或创作者的真实内心状态。

### 2. 可采用的编码框架和数据

| 框架/资源 | 编码内容 | 适合用途与注意事项 |
|---|---|---|
| **效价—唤醒度（valence–arousal）** | 效价从负面/不愉悦到正面/愉悦；唤醒度从平静/低能量到激动/高能量。可用连续刻度评分。 | 适合第一版基线，维度少、便于比较；它概括情绪空间，不会告诉我们具体是哪一种情绪。 |
| **DEAM** ([网站](https://cvml.unige.ch/databases/DEAM/)) | 音乐片段的效价与唤醒度评分，包含随时间变化的标注。 | 可用于连续情绪预测的研究基准。确认标签定义、许可、曲目与时间标注后再使用；与本项目音乐的领域差异要说明。 |
| **GEMS** ([论文](https://doi.org/10.1016/j.cognition.2008.03.001)) | 九个维度：Wonder（惊奇）、Transcendence（超越/崇高）、Tenderness（温柔）、Nostalgia（怀旧）、Peacefulness（平静）、Power（力量感）、Joyful activation（愉悦激活）、Tension（紧张）、Sadness（悲伤）。 | 能表达比正/负面更细的音乐感受，但标注和建模维度较多；先获取并采用论文中的量表说明，不要只凭维度名称自创题目，也不要把不同量表的同名标签直接视为完全等价。 |
| **MTG-Jamendo** ([数据集与任务说明](https://github.com/MTG/mtg-jamendo-dataset)) | 音乐 mood/theme 标签，多为多标签。 | 可做标签预测或检索的补充基准；标签来源和覆盖范围与听众评分不同，不能视为精细、连续的心理测量。 |

推荐起步：先用效价和唤醒度做连续评分，再根据研究问题决定是否增加 GEMS 维度或多标签主题。不同数据集可作外部参照，但不要未经检查就混合其标签。

### 3. 把情绪评分与 MERT、Librosa 对齐

以当前工作流的 **5 秒窗口**作为首轮分析单位，并为每个窗口保留稳定的 `video_id`、音轨/音轨类型、开始与结束时间、原始文件/来源及处理参数。

1. **抽取窗口和表示。** 对音乐主导片段提取 MERT embedding；同时计算 Librosa 特征，例如 RMS/能量、tempo/onset、MFCC/频谱音色、chroma/和声线索。保留 MERT 和 Librosa 两套特征，不要一开始就直接拼接。
2. **取得对齐的情绪标签。** 如果采用逐秒评分，将覆盖该窗口的评分按时间对齐并求平均，同时保留评分者数量和分歧（若有）。如果是本项目人工标注，让评分者听同一窗口并按统一量表评分；记录无法判断或不确定，不要强迫给标签。
3. **先做可解释基线。** 对效价/唤醒度分别用标准化 Librosa 特征训练岭回归；再用 L2 归一化 MERT 向量训练同类回归基线。多标签情绪则使用与标签定义相符的多标签模型。先比较 MERT、Librosa 与简单基线的独立效果，最后才试特征融合。
4. **按视频或歌曲分组验证。** 训练集和测试集必须按整首歌/整条视频分组，不能把同一音频的相邻或重叠窗口随机拆到两边。若样本量允许，使用分组交叉验证；报告测试集相关系数/误差（连续评分）或宏平均 F1、各标签精确率/召回率（分类），并报告标签基线、样本数和不确定性。
5. **听审错误和边界案例。** 试听预测高/低分及预测错误的窗口，检查人声、背景噪声、剪辑、音量、来源差异或 Demucs 伪影是否影响结果；保存人工复核记录。聚类和可视化可探索模式，但不能替代预测评估或情绪标注。

**MERT 如何“匹配”情绪：** MERT 先把窗口编码为向量；它的 768 个维度不是预先命名的情绪轴。只有在带有已知情绪标签的训练样本上拟合预测器，才能检验这些向量是否包含可泛化的情绪信息。用相似度或聚类本身，不能推出某个簇就是“悲伤”或“激昂”。

**Librosa 如何参与：** Librosa 特征提供可检查的声学线索，能帮助描述哪些特征与标签相关，也可作为与 MERT 对照的基线；它们不是情绪标签。相关性也不等于因果。

### 3.1 实际怎样把标签、MERT 和 Librosa 配到一起

关键是所有内容使用同一条音频时间轴和一个明确的窗口主键，而不是靠“向量看起来相似”来匹配标签：

1. **统一音频预处理和窗口。** 对每个参考曲目及项目音频，记录 `dataset_id`、`track_id`、音频路径/哈希、采样率、单/双声道、模型版本、`start_sec`、`end_sec`。用同一组完整 5 秒窗口计算 MERT 向量和 Librosa 特征；以 `(dataset_id, track_id, start_sec, end_sec)` 作为特征表的唯一键。除模型要求的重采样外，不要对不同来源采用不一致的裁切/归一化。
2. **DEAM：时间序列标签按时间聚合。** DEAM 动态效价/唤醒度是逐秒评分。把每个 5 秒窗口与同一 `song_id` 的评分按时间戳相交，在窗口覆盖范围内分别求 valence 和 arousal 均值；标签表和特征表按 `song_id + start/end` 合并。优先动态评分；歌曲级静态评分可用于整曲任务，但不能假装是独立的逐窗真实评分。
3. **MTG-Jamendo：标签是曲目级弱标签。** TSV 中的 `TRACK_ID`、`PATH`、`DURATION`、`TAGS` 描述整首曲目，标签来自上传者。先按官方 train/validation/test 曲目划分，下载并处理相应音频；可将窗口特征按曲目平均后预测整曲 tags，或使用明确的多实例学习。若把同一个曲目标签复制给其所有窗口，要注明这是弱标签，会把整曲标签错误地当成每个局部都成立。绝不能先随机分窗口再拆数据。
4. **GEMS：先收集项目自己的人工评分。** GEMS 不是本地已下载的可直接训练标注集。按九个维度为选定窗口设计问卷/编码表，由多位标注者独立评分；保留 `annotator_id`、窗口时间、评分和不确定度，再按窗口与 MERT/Librosa 特征连接。先检查评分者一致性，再拟合各维度的连续回归模型。
5. **每套标签单独建任务。** 对 DEAM 分别预测效价和唤醒度，对 GEMS 预测各维度，对 MTG-Jamendo 预测 mood/theme 多标签。先分别训练 MERT-only、Librosa-only 基线，再评估融合；不把三种来源/含义不同的标签简单合成一个情绪真值。评估按曲目/视频整组留出，DEAM/MTG 基准成绩也不能直接当作对本项目视频的验证。

这样得到的训练表可以是一行一个窗口，列包含统一主键、`mert_0...mert_767`、Librosa 特征、DEAM 连续分数、可选 GEMS 评分和来源字段；MTG 曲目级任务则另建曲目级特征/标签表。对项目视频的预测只在模型完成留出验证后进行，并保留模型版本与对应时间戳。

### 4. 针对本项目的实际执行顺序

1. 先选取覆盖不同视频、音乐类型和音质的音乐主导窗口；用 `no_vocals` 时记住它是 Demucs 估计的非人声伴奏，并不保证纯音乐，需保留原音轨并抽听检查。
2. 决定标注目标是“音乐表达”还是“听众感受”，写出效价/唤醒度评分说明。由至少两位标注者独立试标一小批窗口，讨论歧义、检查一致性，再修订指南。
3. 将时间戳标签与同窗 MERT/Librosa 特征合并；检查缺失、窗口覆盖、音频版本和标签分布。
4. 建立按视频/歌曲分组的训练与留出测试方案。数据很少时，先报告探索性结果和限制，不要把它包装成已验证的情绪分类器。
5. 同时保留三种互不混淆的监督信号：DEAM 效价/唤醒度、MTG-Jamendo mood/theme 多标签、项目窗口上的 GEMS 人工评分；比较 Librosa、MERT 和后续融合模型，按对应任务分别报告表现。
6. 对外报告时说明标签来自谁、量表是什么、训练/测试如何划分、在哪些音乐/语言/音质上评估、错误与不确定性如何处理。结论应写成“模型在留出视频上对这些评分有某种预测能力”，而不是“模型知道音频真实表达/听众真实感受”。

### 当前状态与下一步建议

已下载 DEAM 全部音频（1,802 首 MP3；压缩包约 1.34 GB，CRC 校验通过并已解压），动态/歌曲级情绪标注、元数据和手册；并核对 1,802 个音频文件 ID 与动态效价标签 ID 全部匹配。动态评分每 500 ms 一列，首个时间点是 15,000 ms；窗口聚合时需按列名中的实际时间筛选，不能假设标签从 0 秒开始。文件位置见 `data/emotion_reference/`。DEAM 音频在 `data/emotion_reference/DEAM/audio/MEMD_audio/`；逐秒 `valence.csv`/`arousal.csv` 在 `data/emotion_reference/DEAM/annotations/annotations/annotations averaged per song/dynamic (per second annotations)/`。MTG-Jamendo 主 mood/theme TSV 为 `data/emotion_reference/MTG-Jamendo/data/autotagging_moodtheme.tsv`，split-0 文件在同目录 `splits/split-0/`；已验证列结构和行数（18,486 曲目总表；官方划分 9,949/3,802/4,231）。MTG-Jamendo 音频尚未下载。整个 `data/emotion_reference/` 目录已加入 `.gitignore`，避免误提交研究数据。GEMS 是要人工施测的编码量表/论文来源，不是可直接下载来训练的带音频标签数据集。

### 资料获取与完整性记录

附件转录复核后，新增的可复现信息是数据下载与校验结果（对话中的工具调用/命令输出本身不作为研究内容归档）：DEAM 音频 ZIP SHA-256 为 `e2814325f573ade8caebc697f0efcfdf7207142b737d72558a416117ca591353`，DEAM 情绪标注 ZIP SHA-256 为 `809b0feb4ba6196c1eda9b2b7b33b892b11fa71457cfceb0d47e832ed1a5f15f`；两个 ZIP 的 CRC 校验均通过。MTG-Jamendo mood/theme 标签 TSV SHA-256 为 `c920e97252586f3e570fa89c4708adcd45a1924dc0e3d3460ac65788257865a8`。数据文件位于 `data/emotion_reference/`，整个目录已在 `.gitignore` 中排除；提交笔记或代码时不要把受许可约束的数据本体一并提交。

已完成 50 首 DEAM 小规模监督基线，并进一步完成全部可用曲目的 5 折分组交叉验证；详细结果和限制见下文。下一步应在项目短视频音乐片段上开展人工评分，检验 DEAM 模型的跨领域迁移，再决定是否训练/应用情绪预测器；MTG 标签仍适合作为独立的曲目级多标签任务，GEMS 仍需按论文量表对项目窗口开展人工标注。项目现有无监督聚类仍不能直接贴上情绪标签。emotion2vec 面向语音情绪表示；分析纯音乐时不应把它当成音乐情绪模型。

### DEAM 基线试验的预期用途

DEAM 用作**带有人类效价/唤醒度评分的监督基准**：在同一首曲目的同一时间窗上配对音频特征与动态评分，训练特征到评分的预测器，并按整首曲目留出测试，以检验 MERT 与 Librosa 是否包含可泛化的评分信息。它不会直接为本项目短视频打标签，也不证明音乐或听众的“真实情绪”。已用固定随机种子对 50 首曲目完成试跑，验证了 ID/时间对齐、特征生成、分组切分和评估流程；小样本结果见下节。对本项目音频的迁移效果仍需另用人工评分验证。

### DEAM 50 首曲目试验结果（探索性）

运行 `scripts/deam_emotion_pilot.py --n-tracks 50 --seed 42`：从 15 秒起取 5 秒、不重叠窗口，每窗汇总 10 个 500 ms 动态评分点；共 300 窗口，按曲目分为 40 首训练和 10 首测试（240/60 窗口）。分别比较标准化的 62 维 Librosa 特征与标准化的 768 维 MERT 均值池化向量，均使用固定岭回归 alpha=10；没有做调参或特征融合。

| 目标 | 方法 | 测试窗 MAE | 测试窗 RMSE | Pearson r |
|---|---|---:|---:|---:|
| 效价 | 训练曲目加权均值基线 | 0.238 | 0.266 | 不适用（常数预测） |
| 效价 | Librosa | **0.160** | **0.207** | 0.605 |
| 效价 | MERT | 0.166 | 0.219 | **0.675** |
| 唤醒度 | 训练曲目加权均值基线 | 0.309 | 0.342 | 不适用（常数预测） |
| 唤醒度 | Librosa | **0.194** | 0.251 | 0.710 |
| 唤醒度 | MERT | 0.208 | **0.243** | **0.731** |

该固定留出集只有 10 首歌，且目前只有一次划分，因此只能说明这次试验中两种表示都优于简单均值基线；Librosa 的窗口误差较低，MERT 的 Pearson 相关较高。不能据此断言哪种方法普遍更好，也不能宣称已得到通用情绪检测器。无调参、无融合、无不确定性分析；DEAM 上的表现不是对项目短视频音频的迁移验证。可复跑脚本为 [`../scripts/deam_emotion_pilot.py`](../scripts/deam_emotion_pilot.py)；详细设计、文件清单和限制见 [`../results/emotion/deam_pilot/README.md`](../results/emotion/deam_pilot/README.md)，指标在 [`../results/emotion/deam_pilot/metrics.csv`](../results/emotion/deam_pilot/metrics.csv)。

### DEAM 全曲库五折分组交叉验证（探索性）

对 1,802 首可用曲目生成 12,964 个 5 秒不重叠窗口；使用打乱的五折 GroupKFold，每首歌只在一个测试折中出现。每折内分别标准化特征并拟合 Ridge（alpha=10），训练损失按曲目等权；比较训练集均值基线、62 维 Librosa、768 维 MERT 和二者拼接。下表是所有 out-of-fold 预测合并后计算的指标，曲目均值列则先对每首歌曲各窗口评分/预测求均值。

| 目标 | 特征 | 窗口 MAE | 窗口 RMSE | 窗口 r | 曲目均值 MAE | 曲目均值 RMSE | 曲目均值 r |
|---|---|---:|---:|---:|---:|---:|---:|
| 效价 | 训练曲目加权均值 | 0.205 | 0.252 | -0.073 | 0.194 | 0.235 | -0.076 |
| 效价 | Librosa | 0.180 | 0.232 | 0.395 | 0.158 | 0.198 | 0.536 |
| 效价 | MERT | 0.167 | 0.217 | 0.528 | 0.138 | 0.173 | 0.674 |
| 效价 | Librosa + MERT | **0.164** | **0.214** | **0.547** | **0.135** | **0.171** | **0.685** |
| 唤醒度 | 训练曲目加权均值 | 0.236 | 0.282 | -0.047 | 0.235 | 0.280 | -0.052 |
| 唤醒度 | Librosa | 0.154 | 0.193 | 0.729 | 0.138 | 0.174 | 0.784 |
| 唤醒度 | MERT | 0.149 | 0.190 | 0.743 | 0.126 | 0.162 | 0.815 |
| 唤醒度 | Librosa + MERT | **0.143** | **0.182** | **0.766** | **0.119** | **0.154** | **0.835** |

在 DEAM 的这套划分与固定岭回归设置下，融合模型在两类目标的汇总指标上最好，MERT 单独模型也优于 Librosa 单独模型；这是数据集内的预测结果，不能解释为 MERT 某些维度本身就是情绪，也不代表对项目短视频有效。仍然只有一个固定五折方案，未调参或估计多次划分的不确定性，且没有项目音频的人工验证。完整报告、逐折指标、每首歌所在折及 out-of-fold 预测见 [`../results/emotion/deam_cross_validation/README.md`](../results/emotion/deam_cross_validation/README.md)、[`metrics.csv`](../results/emotion/deam_cross_validation/metrics.csv)、[`track_folds.csv`](../results/emotion/deam_cross_validation/track_folds.csv) 和 [`test_predictions.csv`](../results/emotion/deam_cross_validation/test_predictions.csv)。缓存特征的评估器是 [`../scripts/deam_emotion_cv.py`](../scripts/deam_emotion_cv.py)；从头重算特征并交叉验证使用 [`../scripts/deam_emotion_pilot.py`](../scripts/deam_emotion_pilot.py) 的 `--all-tracks --cv-folds 5` 选项。

### 参考链接

- DEAM: https://cvml.unige.ch/databases/DEAM/
- GEMS 论文: https://doi.org/10.1016/j.cognition.2008.03.001
- MTG-Jamendo 数据集: https://github.com/MTG/mtg-jamendo-dataset
- MediaEval 音乐情绪与主题识别任务: https://multimediaeval.github.io/2021-Emotion-and-Theme-Recognition-in-Music-Task/
- MERT 模型卡: https://huggingface.co/m-a-p/MERT-v1-95M

## 工具建议：从音频到主题探索

你列出的 **Silero VAD、CLAP、YAMNet/PANNs、BGE-M3、BERTopic 和 HDBSCAN** 已覆盖了主要环节。建议再加上两类关键工具：**ASR 语音转写**（否则难以系统发现说话内容中的主题）和**人工抽样标注/复核**。可选说话人分离、可视化和标注界面，但不必一开始全部部署。

| 阶段 | 推荐工具 | 用途与注意事项 |
|---|---|---|
| 音频读取/特征 | 现有 `librosa`；需要批量音频 I/O 或模型处理时可看 `torchaudio` | 读取、重采样、声学特征及时间轴。优先沿用项目现有的 `librosa`。 |
| 语音区间 | Silero VAD | 找可能有人说话的时间范围；不判断语义或重要性，需抽查噪声、音乐和重叠语音造成的错误。 |
| 中文语音转写（ASR） | **FunASR**；与 **Whisper/faster-whisper** 在小样本上比较 | 生成文字和时间戳，供主题/表达分析。模型对普通话、方言、背景音乐及专有名词效果可能不同；检查并保留不确定转写。选工具前先用目标语料做试验，不预设单一模型最佳。 |
| 语音/音乐/事件线索 | YAMNet 或 PANNs（先择一） | 提供声音事件类别分数，可辅助标记语音、音乐、环境声等；分类分数不是宣传标签。 |
| 音频向量/文本描述相似度 | CLAP | 用于相似声音检索或探索性地比较音频与文字描述；提示词相似度不是事实核验或意图判断。 |
| 文本向量 | BGE-M3 | 将中文/多语言转写片段映射为文本向量，供相似搜索和聚类；分块和长文本处理要保持一致。 |
| 主题探索 | BERTopic | 常用流程是文本 embedding → 降维 → HDBSCAN → c-TF-IDF 主题词表示。BERTopic 是主题建模框架，不等同于单独的向量模型；主题名需研究者复核。 |
| 其他向量聚类 | HDBSCAN | 可在 embedding 上找不同密度的簇，并将部分样本标为噪声/离群点；“噪声点”指聚类算法无法归簇，不等于录音里的噪音。可先用它与 BERTopic 默认流程比较，避免不必要地重复聚类。 |
| 降维可视化 | UMAP | 把高维向量投影成较低维图形，便于观察；图上点的距离是投影结果，不能单独作为聚类正确性的证据。 |
| 人工标注/复核 | Label Studio，或先用 CSV/表格 | 记录时间戳、转写修订、声音类型、主题线索、来源和不确定性。先用小样本检验标注规则是否清楚。 |
| 可选：说话人分段 | pyannote.audio | 在需要分析说话人轮次时使用；它不识别人物真实身份，也不是主题检测器。 |

### 建议的第一版最小组合

**librosa + Silero VAD +（FunASR 与 faster-whisper 的小样本对比后择一）+ BGE-M3 + BERTopic（先用默认聚类流程）+ 人工复核。** 如果还要探索音乐/环境声，再加 CLAP 或 YAMNet/PANNs 中的一种。先分别分析文本主题和音频/声音模式，不急着把所有向量拼接；这样更容易知道每种表示究竟发现了什么。对每个候选主题抽听、检查原始来源，并保留反例及不确定样本。

本清单是建议试验的工具，而非已安装或已接入项目的组件。选型时还应检查模型支持语言、输入时长/采样率要求、许可证、运行资源和隐私/数据处理条件。

## 预训练音频 embedding 是什么？以及如何寻找宣传类音频？

### 预训练音频 embedding（预训练音频向量）

预训练模型先在大量音频上学习声音规律。把一段新音频输入模型后，它会输出一串数字，作为这段音频的向量表示（embedding）。可以把它想成模型做出的“数字化声音速写”：向量接近，表示模型认为两段声音在它学到的某些特征上相似；这**不代表**两段音频主题相同，也不证明它们具有某种意图。

对我们的探索用途，embedding 可以帮助把相似片段放在一起，让研究者寻找反复出现的声音或内容模式。不同工具关注的内容不同：音频向量可能反映声学风格；声音事件模型可能提示语音、音乐或环境声；ASR 转写加文本向量可能帮助发现相似说法、主题和叙事。预训练不代表模型理解了我们的研究问题，也不意味着它针对 Douyin 或宣传内容训练过。它提供的是线索，不是结论或标签。开始时不必急着微调模型；先在人工检查过的样本上评估，确认模型有什么用、会犯什么错，再决定是否需要微调。

### 如何寻找和分析宣传类音频？

“宣传”不是单靠音色、音乐或响度就能判定的声学类别。它通常需要结合**说了什么、如何表达、传播的语境和研究定义**来分析。模型可以协助检索、转写、整理相似片段和发现候选模式，但不应仅凭一个音频 embedding 就自动把内容定性为宣传。

1. **先明确操作性定义和范围。** 写清楚你们在这个研究里如何使用“宣传”一词、研究的主题/时间/地区/语言范围，以及哪些内容不纳入。可以把可能观察的修辞线索作为编码项，例如反复口号、情绪诉求、号召行动、群体对立框架、诉诸权威或紧迫感；这些是需要结合语境核验的分析线索，不是自动判定宣传的充分条件。
2. **建立透明、合规、可复查的语料搜集策略。** 从公开且允许研究使用的材料、合规的平台访问方式、新闻/历史档案或研究语料开始。记录 URL 或来源标识、采集日期、标题/描述、语言、时长、账号/频道可公开的上下文，以及纳入/排除原因；遵守平台条款、版权和隐私要求，不尝试绕过访问限制。只用“已知宣传案例”作种子会造成确认偏差；应同时收集相关主题的多种来源和反例/对照材料。
3. **先做小型试点，再扩大搜集。** 选取不同来源、时长、语言和音质的代表性样本，检查能否获得音频、转写和必要的上下文。避免只挑选听起来最激烈或最符合预期的片段。
4. **用模型生成候选，不让模型代替研究判断。** 可对音频做 ASR（中文/目标语言需检查效果），用关键词和语义搜索找候选转写，再用文本向量/聚类发现重复表达和主题；音频事件/音频 embedding 可补充背景音乐、语气、场景或声音风格的相似性。聚类只说明某种表示空间里的相似，不说明其政治立场或宣传意图。
5. **制定标注指南并人工复核。** 对片段记录时间戳、转写、说话/音乐/混合等声音类型、相关修辞线索、证据出处和不确定性；把“无法判断/需上下文”作为有效标签。尽可能让两位标注者独立标注一部分样本，讨论分歧并修订规则；保留反例，并在独立样本上检查一致性。
6. **验证而不是只看聚类图。** 听/读每个簇的代表样本和边界样本，检查是否其实按语言、音质、音乐、频道或重复转载聚在一起。与非宣传或其他类型的相近内容比较，并报告采样局限、误差和不确定性。

因此，一个稳妥的发现流程是：**明确研究定义 → 合规、平衡地收集并记录来源 → ASR/声学分析生成候选 → 向量化与聚类辅助探索 → 人工按指南复核和标注 → 用对照样本验证解释**。聚类可以帮助发现潜在主题和模式；要声称某片段属于宣传，仍需透明的定义、内容/语境证据和可复查的人工判断。

## One-sentence summary

Use coarse structure boundaries as hypotheses, not labels; detect and review speech/music/events first, then make timestamped task-appropriate windows and vectorize them while preserving the original audio and uncertainty about what counts as meaningful foreground.

## Work journal / log

| Timestamp (UTC) | Work completed / current understanding |
|---|---|
| 2026-10-03 00:13–00:22 | Inspected the existing `results/` outputs for the Billie Eilish trial: feature summaries, chroma profiles, Demucs vocal/accompaniment stems, Basic Pitch note events, beat tracking, key candidates, and coarse librosa structural segmentation. Discussed moving toward Douyin audio, short-window features/vectorization/clustering, and the need to distinguish speech/content from background and technical noise. |
| 2026-10-03 00:22 | Reviewed the segmentation code and CSV. Confirmed it uses chroma self-similarity with agglomerative segmentation fixed at `k=8`; boundaries are exploratory musical-structure candidates, not short windows, semantic phases, or speech/background labels. Noted variable span lengths and that the last saved boundary (about 189.40 s) precedes the approximately 205.93 s track end. Added these caveats and a practical foreground/cleanup workflow here. |
| 2026-10-03 00:29 | Added a current-state workflow diagram distinguishing implemented Billie Eilish trial analyses from planned Douyin steps, plus a comparison of common handcrafted, pretrained audio, speech, speaker, and text-vectorization tools. Clarified that no segment-level vector matrix or clustering is implemented yet. |
| 2026-10-03 00:32 | Added a Chinese explanation of pretrained audio embeddings and a discovery workflow for propaganda-related audio, emphasizing transparent definitions, compliant and balanced corpus collection, model-assisted candidate discovery, human review, contextual evidence, and validation against comparison samples. |
| 2026-10-03 00:36 | Added a Chinese tool-selection guide mapping VAD, ASR, audio event models, audio/text embeddings, topic modeling, clustering, visualization, and human review to their roles. Included a minimal starter stack and clarified that HDBSCAN noise points are not acoustic noise. |
| 2026-10-03 01:08 | For the 国庆 test video, decided to compare three analysis inputs: the original mixed audio, Demucs vocals stem, and Demucs no_vocals/accompaniment stem. The separation is an experimental branch, not ground truth; downstream quality checks and handcrafted features should run on all three with stem identity retained. |
| 2026-10-03 01:13 | Added an executable audio manifest step to the test notebook for the three versions, including paths, sample/video identifiers, source fields, Demucs settings/version, audio metadata, file sizes, checksums, and window parameters. Unknown source URL/retrieval time remain blank for later completion. |
| 2026-10-03 01:15 | Generated and validated `results/test_video/audio_manifest.csv` with one row each for `original_mix`, `vocals`, and `no_vocals`. The manifest is intended to support future batch runs by providing a consistent input inventory and traceable metadata/paths for each video audio variant. |
| 2026-10-03 01:22 | Documented the exact composition and rationale of the 62 handcrafted features in `test.ipynb`; added the clean notebook's music-analysis workflow for all three audio variants (beat tracking, chroma/key candidates, five-second pitch-class timeline, coarse segmentation/recurrence, and cross-stem DTW), plus per-stem Basic Pitch MIDI/note-event cells. Clarified MERT/CLAP and other model/cluster stages that remain unrun. |
| 2026-10-03 01:23 | Smoke-tested the new Librosa music-analysis cell against the three existing audio versions using a temporary output directory. Beat/chroma/key candidates, segment intervals, recurrence plots, and cross-stem DTW outputs were generated successfully; Basic Pitch transcription was not run. |
| 2026-10-03 01:27 | Added an optional MERT-v1-95M embedding cell using the model-card sample rate/context, three stem inputs, timestamped embedding index, and explicit Hugging Face download/remote-code notice. Added a CLAP prompt-similarity example and clarified which embedding/event/ASR/clustering steps remain unrun. |
| 2026-10-03 01:42 | Confirmed the test-video run completed: 246 MERT vectors (768 dimensions; 82 windows per original_mix/vocals/no_vocals), Librosa beat/chroma/key/structure/DTW outputs, and Basic Pitch CSV/MIDI outputs for all three versions. Added a results interpretation/review section to `test.ipynb` and a reusable optional MERT vectorization section to the master notebook. Basic Pitch event CSV rows contain variable-length pitch-bend fields, so their event columns need careful parsing rather than assuming a rectangular table. |

| 2026-10-03 03:52 | Promoted the reusable analysis workflow to `notebooks/master.ipynb`, migrating configuration-driven input/output paths, per-variant manifest and checksums, quality checks/waveforms, 62-D window features, multi-variant Librosa analyses, MERT indexing, and per-variant Basic Pitch. Reset `test.ipynb` to a scratch template; retained the completed sample outputs and did not migrate the opt-in Douyin downloader. Updated project entry-point documentation. |

| 2026-10-03 03:53 | Removed all yt-dlp/online batch-download cells and their setup from the master notebook at the user's request. The master now takes a locally supplied video or audio file; video audio is extracted locally with FFmpeg and both source-media and analysis-audio provenance are recorded. |

| 2026-10-03 03:56 | Created the Git-ignored `uploads/` folder for user-supplied source media; the master notebook now reads `AUDIO_UPLOADS_DIR` (defaulting to `uploads/`) so it can target a mounted persistent volume. Documented the limitation that the current `overlay` filesystem is not a durability guarantee and recommended provider-backed persistent disk/object storage plus backup for irreplaceable originals. |

| 2026-10-03 04:01 | Removed the newly created `uploads/` folder after confirming the existing `video/` directory is the project’s media drop location. Updated the master notebook and storage guidance to use `video/` by default, with `MEDIA_SOURCE_DIR` for an externally mounted persistent volume. |
| 2026-10-03 04:10 | Ran the master pipeline on `video/-国庆111572550806.mp4` (29.91 s, 44.1 kHz stereo AAC): extracted WAV, Demucs original/vocals/no_vocals, 3-row manifest and quality checks, 36 rows of 62 handcrafted window features, Librosa tempo/chroma/key/segments/DTW, MERT (36×768), and Basic Pitch MIDI/CSV for all three variants. Results are in `results/master/-国庆111572550806/`. These are candidate acoustic/music estimates only; no ASR, propaganda/content classification, or clustering was run. Basic Pitch logged missing GPU libraries but completed successfully. |
| 2026-10-03 04:17 | Ran exploratory K-means clustering on the existing 36 timestamped windows, separately using standardized 62-D handcrafted features and L2-normalized 768-D MERT embeddings. Tested k=2–6; saved k=3 assignments, silhouette sweep, stem composition, and PCA plots under `results/master/-国庆111572550806/clustering/`. Handcrafted k=3 separated the three Demucs variants exactly; MERT k=3 produced one vocals-only cluster and two mixed-stem clusters. Because all windows come from one video, overlap in time, and include related separated stems, these results mainly diagnose representation/stem differences and are not evidence of semantic or propaganda categories. |
| 2026-10-03 04:21 | Ran a no_vocals-only K-means follow-up using its 12 MERT windows (k=2–6), and added the reproducible section to `master.ipynb`. k=2 had silhouette 0.489, but produced one singleton cluster containing only the final 27.5–29.91 s, 2.41 s partial window (padded for MERT), with the other 11 windows in the other cluster. This score is therefore likely driven by the short padded tail/outlier and must not be interpreted as a meaningful discovered audio category. |

| 2026-10-03 04:34 | Added automatic FFmpeg trailing-silence detection to the master workflow: detect at −40 dB with a 0.25 s detector minimum, trim only when the final silence is at least 0.5 s, preserve the full source, and record source/analysis audio hashes separately. Re-ran the 29.91 s video after trimming the final 0.907 s to an analysis duration of 29.001 s. Recomputed stems, manifest, features, music analysis, 30 full-window MERT vectors, Basic Pitch, and clustering. Excluded padded partial windows from MERT/clustering; no-vocals-only k=2 silhouette is now 0.274 with 5 windows per group and interleaved overlapping timestamps, so no clear contiguous phase split is established. |

| 2026-10-03 04:43 | Scanned all 15 audio/video media files under project audio/video/results, excluding paths containing Billie Eilish. FFmpeg detected qualifying trailing silence (≥0.5 s at −40 dB) in 5 files: current source audio (0.907 s), two untrimmed Demucs stems for that sample (~0.90 s each), a prior test-video vocals stem (3.272 s), and the source MP4 audio (~0.907 s). Preserved every source and generated separate 44.1 kHz stereo WAV copies under `results/trailing_silence_cleanup/`; audit report includes durations and SHA-256 hashes. No qualifying tail silence in other 10 files. |

| 2026-10-03 04:46 | Ran cross-video no-vocals MERT clustering on 98 complete 5-second windows from three non-Billie samples (7, 10, and 81 windows); excluded partial padded windows and used a 7-window-per-video balanced comparison. The balanced k=3 result separated exactly by source video, demonstrating likely sample/video identity confounding rather than semantic categories; balanced silhouette k=3=0.365, k=4=0.384. Saved assignments, scores, sample composition, and PCA plot under `results/corpus_clustering/no_vocals_mert/`; added repeatable corpus-level cell to master. |

| 2026-10-03 04:50 | Removed the ten Billie Eilish source tracks, their identifiable derived Demucs/Librosa/Basic Pitch outputs (including the large recurrence matrix), stale Billie notebook checkpoints, and all embedded execution outputs from the legacy notebook; replaced its hardcoded sample paths with generic placeholders. Historical research notes were retained as requested. |

| 2026-10-03 04:57 | Updated research notes with the current prioritized analyses: emotion2vec speech-emotion representations, Librosa interpretable acoustic features, MERT/Librosa cross-video music similarity, and optional CLAP audio-text retrieval. Explicitly deferred VAD+ASR text semantics and broad sound-event classification. Added a cross-video music-similarity workflow covering windowing, complete-window filtering, no-vocals caveats, cosine nearest neighbors, source/video balancing, interpretation controls, listening validation, and the current finding that three-video balanced clusters separated by source rather than shared music. |
| 2026-10-03 05:06–05:10 | Added a concrete alignment plan for DEAM valence/arousal, all nine GEMS dimensions, MTG-Jamendo mood/theme tags, timestamped MERT windows, and Librosa features. Downloaded and verified DEAM annotations/metadata/manual/audio (1,802 MP3s; archive CRC passed) and MTG-Jamendo mood/theme metadata, vocabulary, split-0 train/validation/test files, README, and license. DEAM labels require timestamp aggregation; MTG tags remain track-level weak labels; GEMS still requires project annotation. MTG audio was intentionally not downloaded pending scope approval. |
| 2026-10-03 05:18–05:20 | Clarified DEAM's role as a supervised benchmark and completed a fixed-seed 50-track pilot: aligned 300 non-overlapping 5 s windows from 15 s onward to DEAM's 500 ms valence/arousal ratings; compared 62-D Librosa and 768-D MERT Ridge baselines using a 40/10 track-grouped split. Both beat the track-weighted-mean baseline on this single small holdout; results are exploratory and do not validate transfer to project clips. Added a reproducible script and pilot report, and refreshed the repository README with clickable English/Chinese sections, tech stack, usage, dataset, and interpretation caveats. |
| 2026-10-03 05:25 | Refined the bilingual GitHub README using `rclone/rclone` as a layout reference: centered project identity, quick language navigation, clear feature/tech-stack/quick-start sections, direct experiment links, and a friendly visual tone without copying its branding or content. Verified local Markdown links and cross-checked the DEAM pilot figures in the notebook against the saved metrics/configuration. |
| 2026-10-03 05:29–06:05 | Completed the requested follow-up: extracted features for all 1,802 usable DEAM tracks (12,964 non-overlapping windows) and evaluated shuffled, track-grouped five-fold out-of-fold predictions for the track-weighted-mean baseline, Librosa, MERT, and fused features. Fused features had the best pooled DEAM metrics for both valence and arousal; detailed scores and limits are recorded in `results/emotion/deam_cross_validation/`. Added `scripts/deam_emotion_cv.py` to re-evaluate cached features without repeating extraction. Rebranded the bilingual GitHub README as Pika-Audio, Pikachu-inspired but explicitly unofficial and without character artwork. |

Add entries with an ISO 8601 UTC timestamp when meaningful project work is completed. Record what was actually done, the data/model involved, and important caveats; do not treat this journal as a substitute for experiment metadata or reproducible logs.
| 2026-10-03 03:27 | Updated the batch yt-dlp downloader after Douyin returned HTTP 403 / “Fresh cookies are needed”: documented safe local Netscape cookie refresh steps, added a first-URL session check with immediate stop on auth/403 errors, and cleared the notebook’s stale failure output. MP3 paths and the CSV manifest remain unchanged; cookies are never written into notebook code or manifest. |
| 2026-10-03 03:32 | Added direct yt-dlp browser-cookie support to the batch audio downloader (`cookiesfrombrowser`), defaulting to Chrome with configurable browser/profile and retaining local Netscape cookie-file mode. Documented the remote-kernel limitation (browser profile must be on the kernel host and browser should be closed), cleared the stale FileNotFoundError output, and verified browser/file option wiring with mocked yt-dlp. |
| 2026-10-03 03:34 | Confirmed the notebook-kernel host has no Chrome/Chromium/Firefox/Edge browser executable or profile, so direct `cookiesfrombrowser` cannot access the user's desktop session in this remote environment. Added browser-profile preflight checks and an explicit remote-kernel explanation with secure cookie-file/local-kernel alternatives; cleared stale notebook error output. |
| 2026-10-03 03:35 | Reverted the batch downloader to local Netscape cookie-file mode at the user's request; removed browser-profile detection/configuration, restored `COOKIE_FILE` validation and yt-dlp `cookiefile` wiring, updated instructions, and cleared the obsolete cell output. |
| 2026-10-03 03:44 | Switched the test notebook to the uploaded `jiayou_guoqing.mp4`, copied it into the ignored `video/` folder, and made batch downloading opt-in so its cookie check does not block local analysis. Extracted audio, ran Demucs, generated manifest/quality checks, 62-D window features, Librosa analyses, MERT (27×768; 9 windows per stem), and Basic Pitch outputs. Added a result interpretation section with uncertainty caveats; no speech/content classification was run. |
